# Training HMM Models with CmdStanPy

This notebook is dedicated to training **HMM** models for S = 2, 3, and 4.

**Note:**

To train standard **VD-HMM** models, please refer to notebook 2a.
If you are reproducing results using notebook cells, make sure to run both this notebook and notebook 2a concurrently.

If you want to run training outside of notebook run:

```bash
  uv run scripts/main.py --model hmm --seed {seed} --state {number of states} --chains {number of chains} --parallel-chains {number of parallel chains}

```

Look at the `main.py` file in the `scripts` folder for full documentation


In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path("..").resolve()))

import pickle
import numpy as np
from helpers import ModelData
import cmdstanpy

SEED = 42  # set seed to train with correct indices (change, if desired)
SET_ORIGINAL_INDICES = False  # if set to true, the original paper indices are selected

print(f"CmdStanPy Version: {cmdstanpy.__version__}")
print(f"CmdStan Path: {cmdstanpy.cmdstan_path()}")

In [ ]:
# Load processed data (with seed)
from constants import PROCESSED_DATA_FOLDER

if SET_ORIGINAL_INDICES:
    data_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"
else:
    data_path = PROCESSED_DATA_FOLDER / f"processed_data_original.pkl"


model_data = ModelData.from_pickle(data_path)

print(model_data.summary())

In [ ]:
from constants import FITTED_MODEL_FOLDER, STAN_MODEL_FOLDER
from helpers import prepare_stan_data


def train_model_cmdstan(
    model_data,
    S,
    model_name="hmm",
    chains=2,
    parallel_chains=2,
    iter_warmup=500,
    iter_sampling=500,
    seed=42,
    adapt_delta=0.8,
    max_treedepth=10,
):
    """
    Trains a model with CmdStanPy.

    Parameters:
    -----------
    model_data : ModelData
        Data for training
    S : int
        Number of hidden states (2-5)
    model_name : str
        'vdhmm' or 'hmm'
    chains : int
        Number of MCMC chains
    parallel_chains : int
        Number of parallel chains (uses parallel_chains CPU cores)
    iter_warmup : int
        Warmup iterations
    iter_sampling : int
        Sampling iterations (post-warmup)
    seed : int
        Random seed
    adapt_delta : float
        Stan adapt_delta parameter (0.8-0.99, higher = more conservative)
    max_treedepth : int
        Stan max_treedepth parameter

    Returns:
    --------
    cmdstanpy.CmdStanMCMC : Fit object
    """
    assert model_name in ["vdhmm", "hmm"], f"Invalid model_name: {model_name}"
    assert S in range(2, 6), "S must be between 2 and 5"

    # Prepare data
    stan_data = prepare_stan_data(model_data, S)

    # Model file
    model_file = STAN_MODEL_FOLDER / f"{model_name}.stan"
    if not model_file.exists():
        raise FileNotFoundError(f"Stan model not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states (CmdStanPy)")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")
    print(f"\nConfiguration:")
    print(f"  Chains: {chains}")
    print(f"  Parallel chains: {parallel_chains}")
    print(f"  Warmup iterations: {iter_warmup}")
    print(f"  Sampling iterations: {iter_sampling}")
    print(f"  Total iterations: {iter_warmup + iter_sampling}")
    print(f"  Seed: {seed}")
    print(f"  Adapt delta: {adapt_delta}")
    print(f"  Max treedepth: {max_treedepth}")

    # Compile model
    print(f"\nCompiling model...")
    model = cmdstanpy.CmdStanModel(stan_file=str(model_file))
    print(f"✓ Model compiled")

    # Sample
    print(f"\nSampling...")
    fit = model.sample(
        data=stan_data,
        chains=chains,
        parallel_chains=parallel_chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
        show_progress=True,
        inits=0,
    )

    print(f"\n✓ Sampling complete!")

    # Save model
    output_path = FITTED_MODEL_FOLDER / f"{model_name}_{S}_cmdstan.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "summary": fit.summary(),
            },
            f,
        )

    print(f"✓ Model saved to {output_path}")

    # Diagnostics
    print(f"\n{'-'*60}")
    print("Diagnostics:")
    print(f"{'-'*60}")
    print(fit.diagnose())

    # Summary statistics
    print(f"\n{'-'*60}")
    print("Summary (first 20 parameters):")
    print(f"{'-'*60}")
    summary_df = fit.summary()
    print(summary_df.head(20))

    return fit


print("✓ Training function defined (CmdStanPy)")

## Training Configuration

**Paper-Standard:** 2 chains × 1000 iterations (500 warmup + 500 sampling)

**For quick testing:** 2 chains × 200 iterations (100 warmup + 100 sampling)

In [ ]:
# Training Settings
SEED = 42
CHAINS = 4
PARALLEL_CHAINS = 4  # Uses 4 CPU cores in parallel
ITER_WARMUP = 1000  # Paper: 500, Quick test: 100
ITER_SAMPLING = 1000  # Paper: 500, Quick test: 100
ADAPT_DELTA = 0.95  # 0.8-0.95, higher if divergent transitions
MAX_TREEDEPTH = 10  # 10-15

np.random.seed(SEED)

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {CHAINS}")
print(f"  Parallel chains: {PARALLEL_CHAINS}")
print(f"  Warmup iterations: {ITER_WARMUP}")
print(f"  Sampling iterations: {ITER_SAMPLING}")
print(f"  Total iterations: {ITER_WARMUP + ITER_SAMPLING}")
print(f"  Total posterior samples: {CHAINS * ITER_SAMPLING}")

## Training HMM Models (S=2, 3, 4)

Standard Hidden Markov Models with time-independent transition probabilities.

In [ ]:
# Dictionary to store all models
trained_models = {}

# HMM Training for S=2 to S=4
for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model_cmdstan(
            model_data=model_data,
            S=S,
            model_name="hmm",
            chains=CHAINS,
            parallel_chains=PARALLEL_CHAINS,
            iter_warmup=ITER_WARMUP,
            iter_sampling=ITER_SAMPLING,
            seed=SEED,
            adapt_delta=ADAPT_DELTA,
            max_treedepth=MAX_TREEDEPTH,
        )

        trained_models[f"hmm_{S}"] = fit
        print(f"\n✓✓✓ HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("HMM Training Complete!")
print("=" * 60)

## Training Summary


In [ ]:
# Summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {FITTED_MODEL_FOLDER}")

# List saved models
saved_models = sorted(FITTED_MODEL_FOLDER.glob("hmm_*_cmdstan.pkl"))
print(f"\nSaved HMM model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")